# Équation de diffusion de la chaleur 2D

$$\frac{\partial u}{\partial t}=\alpha\left(\frac{\partial^2 u}{\partial x^2}+\frac{\partial^2 u}{\partial y^2}\right)$$

- Domaine : le carré $[0,1]^2$
- Bord : $u=0$
- Départ : $\sin(\pi x)\,\sin(\pi y)$
- Solution exacte : $\sin(\pi x)\sin(\pi y)\,e^{-2\pi^2\alpha t}$ (sert à vérifier le calcul)

## 1. Vérifier les bibliothèques

In [ ]:
import numpy, scipy, matplotlib, sympy
for m in (numpy, scipy, matplotlib, sympy):
    print(m.__name__, m.__version__)
# Si une bibliothèque manque : %pip install numpy scipy matplotlib sympy

## 2. Le script `chaleur2d.py`
Quatre blocs : **SymPy** (vérification de la solution exacte) · **NumPy** (grille) · **SciPy** (résolution, Crank–Nicolson) · **Matplotlib** (figure).

In [ ]:
%%writefile chaleur2d.py
"""chaleur2d.py - Equation de la chaleur 2D sur [0,1]^2, u = 0 au bord.
Bloc 1 SymPy | Bloc 2 NumPy | Bloc 3 SciPy | Bloc 4 Matplotlib"""
import numpy as np
import sympy as sp
import scipy.sparse as sps
from scipy.sparse.linalg import splu
import matplotlib.pyplot as plt

ALPHA = 0.1      # diffusivite (a changer pour l'essai)
N = 50           # points interieurs par direction (a doubler pour l'essai)
T_FIN = 0.5
DT = 0.005

# ---------- 1 . SymPy : la solution exacte verifie-t-elle l'EDP ? ----------
x, y, t, a = sp.symbols("x y t a", positive=True)
u = sp.sin(sp.pi*x)*sp.sin(sp.pi*y)*sp.exp(-2*sp.pi**2*a*t)
res = sp.diff(u, t) - a*(sp.diff(u, x, 2)
                         + sp.diff(u, y, 2))
print("Résidu de l'EDP pour la solution exacte :", sp.simplify(res))   # donne 0
u_ex = sp.lambdify((x, y, t), u.subs(a, ALPHA), "numpy")

# ---------- 2 . NumPy : grille (points interieurs, u = 0 au bord) ----------
n = N
h = 1.0/(n + 1)
xg = np.linspace(h, 1 - h, n)
X, Y = np.meshgrid(xg, xg,
                   indexing="ij")

# ---------- 3 . SciPy : Laplacien (Kronecker) + Crank-Nicolson ----------
T = sps.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n))/h**2
I = sps.identity(n)
L = sps.kron(I, T) + sps.kron(T, I)
Id = sps.identity(n*n)
A = Id - 0.5*DT*ALPHA*L
B = Id + 0.5*DT*ALPHA*L
lu = splu(A.tocsc())

nsteps = int(round(T_FIN/DT))
U = u_ex(X, Y, 0.0).ravel()
snaps = {0.0: U.reshape(n, n).copy()}
temps, erreurs = [], []
for k in range(1, nsteps + 1):
    U = lu.solve(B @ U)                    # un pas
    tk = k*DT
    erreurs.append(np.max(np.abs(U - u_ex(X, Y, tk).ravel())))
    temps.append(tk)
    for ts in (0.25, 0.5):
        if abs(tk - ts) < 1e-12:
            snaps[ts] = U.reshape(n, n).copy()

print(f"Erreur maximale sur [0, {T_FIN}] : {max(erreurs):.2e}")

# ---------- 4 . Matplotlib ----------
fig, axs = plt.subplots(1, 4, figsize=(15, 3.6),
                        gridspec_kw={"width_ratios": [1, 1, 1, 1.2]})
for ax, (ts, f) in zip(axs[:3], sorted(snaps.items())):
    m = ax.pcolormesh(X, Y, f, cmap="inferno", vmin=0, vmax=1, shading="auto")
    ax.set_title(f"t = {ts:.2f}")
    ax.set_aspect("equal")
fig.colorbar(m, ax=axs[:3], label="température", shrink=0.9)
axs[3].semilogy(temps, erreurs)
axs[3].set_xlabel("t"); axs[3].set_ylabel("erreur max")
axs[3].set_title("numérique vs analytique")
fig.savefig("chaleur2d.png", dpi=130, bbox_inches="tight")
print("Figure enregistrée : chaleur2d.png")
plt.show()

## 3. Lancer le script

In [ ]:
%run chaleur2d.py

Résultat attendu :
```
Résidu de l'EDP pour la solution exacte : 0
Erreur maximale sur [0, 0.5] : 1.13e-04
Figure enregistrée : chaleur2d.png
```
La température s'étale et décroît ; l'erreur maximale est de l'ordre de 1e-4.

## 4. À essayer : doubler N ou changer ALPHA

In [ ]:
import numpy as np, scipy.sparse as sps
from scipy.sparse.linalg import splu

def resoudre(N, alpha, dt=0.005, t_fin=0.5):
    h = 1.0/(N + 1)
    xg = np.linspace(h, 1 - h, N)
    X, Y = np.meshgrid(xg, xg, indexing="ij")
    ex = lambda t: (np.sin(np.pi*X)*np.sin(np.pi*Y)*np.exp(-2*np.pi**2*alpha*t)).ravel()
    T = sps.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(N, N))/h**2
    L = sps.kron(sps.identity(N), T) + sps.kron(T, sps.identity(N))
    Id = sps.identity(N*N)
    lu = splu((Id - 0.5*dt*alpha*L).tocsc())
    B = Id + 0.5*dt*alpha*L
    U = ex(0.0); err = 0.0
    for k in range(1, int(round(t_fin/dt)) + 1):
        U = lu.solve(B @ U)
        err = max(err, np.max(np.abs(U - ex(k*dt))))
    return err

for N in (50, 100):
    print(f"N = {N:<4} ALPHA = 0.1   erreur max = {resoudre(N, 0.1):.2e}")
for al in (0.05, 0.2):
    print(f"N = 50   ALPHA = {al:<5} erreur max = {resoudre(50, al):.2e}")

Quand N double, l'erreur est divisée par environ 4 (ordre 2 en espace).

## 5. Télécharger la figure

In [ ]:
try:
    from google.colab import files
    files.download("chaleur2d.png")
except ImportError:
    print("Figure enregistrée dans le dossier courant : chaleur2d.png")